# 01 — Exploratory Data Analysis

**Dataset:** [Mercari Price Suggestion Challenge](https://www.kaggle.com/c/mercari-price-suggestion-challenge)  
**Goal:** Understand price distributions, category structure, brand landscape, and missing-value patterns before building the LightGBM pricing model.

---

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config

# Plotting defaults
sns.set_theme(style="whitegrid", palette="viridis", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

%matplotlib inline

## 1. Load raw training data

In [ ]:
df = pd.read_csv(config.TRAIN_FILE, sep="\t")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
# Drop zero-price rows (same as features.py)
df = df[df["price"] > 0].copy()
print(f"After removing price == 0: {df.shape[0]:,} rows")

## 2. Price distribution (log scale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price
axes[0].hist(df["price"], bins=100, color="#4c72b0", edgecolor="white", alpha=0.85)
axes[0].set_title("Price Distribution (raw)", fontweight="bold")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Count")
axes[0].axvline(df["price"].median(), color="#c44e52", ls="--", label=f'Median = ${df["price"].median():.0f}')
axes[0].legend()

# Log-transformed price
log_price = np.log1p(df["price"])
axes[1].hist(log_price, bins=100, color="#55a868", edgecolor="white", alpha=0.85)
axes[1].set_title("log₁ₚ(Price) Distribution", fontweight="bold")
axes[1].set_xlabel("log(1 + price)")
axes[1].set_ylabel("Count")
axes[1].axvline(log_price.median(), color="#c44e52", ls="--", label=f"Median = {log_price.median():.2f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "price_distribution.png", bbox_inches="tight")
plt.show()

print(f"Price stats:\n{df['price'].describe().to_string()}")

## 3. Price by category (top 10 categories — boxplot)

In [ ]:
# Parse main category
df["category_main"] = (
    df["category_name"]
    .fillna("unknown")
    .str.split("/").str[0]
)

top_10_cats = df["category_main"].value_counts().head(10).index.tolist()
df_top = df[df["category_main"].isin(top_10_cats)].copy()

# Order by median price (descending)
cat_order = (
    df_top.groupby("category_main")["price"]
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=df_top,
    x="category_main",
    y="price",
    order=cat_order,
    showfliers=False,
    palette="viridis",
    ax=ax,
)
ax.set_title("Price by Category (Top 10 by listing count)", fontweight="bold")
ax.set_xlabel("Main Category")
ax.set_ylabel("Price ($)")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "price_by_category.png", bbox_inches="tight")
plt.show()

## 4. Missing values heatmap

In [ ]:
missing = df.isnull().sum().to_frame(name="missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
missing = missing.sort_values("missing_count", ascending=False)
print(missing.to_string())
print()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    df.isnull().T,
    cbar=False,
    cmap="YlOrRd",
    yticklabels=True,
    ax=ax,
)
ax.set_title("Missing Values Heatmap (yellow = missing)", fontweight="bold")
ax.set_xlabel("Row index")

plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "missing_values.png", bbox_inches="tight")
plt.show()

## 5. Correlation of numeric features with price

In [ ]:
# Build numeric features for correlation analysis
df["log_price"] = np.log1p(df["price"])
df["desc_length"] = df["item_description"].fillna("").replace("No description yet", "").str.len()
df["name_length"] = df["name"].fillna("").str.len()
df["shipping_flag"] = df["shipping"]

numeric_cols = ["price", "log_price", "shipping_flag", "item_condition_id", "desc_length", "name_length"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Feature Correlation Matrix", fontweight="bold")

plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "correlation_matrix.png", bbox_inches="tight")
plt.show()

print("\nCorrelations with price:")
print(corr["price"].sort_values(ascending=False).to_string())

## 6. Top 20 brands by listing count

In [ ]:
brand_counts = (
    df["brand_name"]
    .fillna("(no brand)")
    .value_counts()
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 6))
brand_counts.plot.barh(ax=ax, color="#4c72b0", edgecolor="white")
ax.set_title("Top 20 Brands by Listing Count", fontweight="bold")
ax.set_xlabel("Number of Listings")
ax.set_ylabel("Brand")
ax.invert_yaxis()  # highest count on top

plt.tight_layout()
plt.savefig(config.OUTPUTS_DIR / "top_brands.png", bbox_inches="tight")
plt.show()

print(brand_counts.to_string())

## 7. Summary statistics

In [ ]:
summary = df[["price", "log_price", "shipping", "item_condition_id", "desc_length", "name_length"]].describe().T
summary["missing"] = df[["price", "log_price", "shipping", "item_condition_id", "desc_length", "name_length"]].isnull().sum()
summary["missing_pct"] = (summary["missing"] / len(df) * 100).round(2)

print("=" * 60)
print("Summary Statistics")
print("=" * 60)
summary

In [ ]:
# Additional categorical summaries
print(f"Unique categories (main) : {df['category_main'].nunique()}")
print(f"Unique brands            : {df['brand_name'].nunique()}")
print(f"Shipping = free (1)      : {df['shipping'].sum():,} ({df['shipping'].mean()*100:.1f}%)")
print(f"Condition distribution   :")
print(df["item_condition_id"].value_counts().sort_index().to_string())

---

**Next:** `02_feature_engineering.ipynb` → run `src/features.py` pipeline and validate outputs.